In [1]:

from rdflib import Graph, Namespace, Literal, URIRef
from rdflib.namespace import RDF, RDFS, XSD, SDO, OWL
import pandas as pd
import os
from dotenv import set_key, load_dotenv
from spacy_llm.util import assemble
from langchain.vectorstores import FAISS
from langchain.embeddings import HuggingFaceEmbeddings

/opt/anaconda3/envs/lab3_py39/lib/python3.9/site-packages/google/api_core/_python_version_support.py:246: FutureWarning: You are using a non-supported Python version (3.9.25). Google will not post any further updates to google.api_core supporting this Python version. Please upgrade to the latest Python version, or at least Python 3.10, and then update google.api_core.
  warnings.warn(message, FutureWarning)
/opt/anaconda3/envs/lab3_py39/lib/python3.9/site-packages/google/auth/__init__.py:54: FutureWarning: You are using a Python version 3.9 past its end of life. Google will update google-auth with critical bug fixes on a best-effort basis, but not with any other fixes or features. Please upgrade your Python version, and then update google-auth.
  warnings.warn(eol_message.format("3.9"), FutureWarning)
/opt/anaconda3/envs/lab3_py39/lib/python3.9/site-packages/google/oauth2/__init__.py:40: FutureWarning: You are using a Python version 3.9 past its end of life. Google will update google-a

In [2]:
# Load graph
g = Graph()
g.parse('./graphs/KG_base_dance_merged.ttl', format='turtle')
print(f"Loaded {len(g)} triples from graph")

Loaded 5894 triples from graph


In [3]:
query_instrument = """
  SELECT ?instrument ?instrumentLabel ?instrumentDescription WHERE {
  SERVICE wikibase:label { bd:serviceParam wikibase:language "[AUTO_LANGUAGE],mul,en". }
  ?instrument wdt:P31 wd:Q34379.
}
LIMIT 10000
  """

query_danceStyles = """
    SELECT ?danceStyle ?danceStyleLabel ?danceStyleDescription WHERE {
    SERVICE wikibase:label { bd:serviceParam wikibase:language "[AUTO_LANGUAGE],mul,en". }
        ?danceStyle wdt:P31 wd:Q107357104.

}
LIMIT 3500
  """

query_dance_practitioner = """
    SELECT ?musicalGroup ?musicalGroupLabel ?musicalGroupDescription WHERE {
    SERVICE wikibase:label { bd:serviceParam wikibase:language "[AUTO_LANGUAGE],mul,en". }
        ?musicalGroup wdt:P31 wd:Q215380.

}
LIMIT 20000
"""

query_dance_music_genre = """
    SELECT ?music_genre ?music_genreLabel ?music_genreDescription WHERE {
    SERVICE wikibase:label { bd:serviceParam wikibase:language "[AUTO_LANGUAGE],mul,en". }
        ?music_genre wdt:P31 wd:Q188451.
}
LIMIT 10000
"""

In [4]:
import requests

def query_wikidata(query):
  url = 'https://query.wikidata.org/sparql'
  query = query

  headers = {
      'User-Agent': 'KG/Assignment01',
      'Accept': 'application/sparql-results+json'
  }

  response = requests.get(url, params={'query': query, 'format': 'json'}, headers=headers)
  data = response.json()
  return data


In [5]:
# Create vector store
def create_vector_store(embedding_model, texts=None, metadatas=None):
    if texts is None or len(texts) == 0:
        raise ValueError("Text data cannot be empty when initializing the vector store.")

    # Add embeddings and metadata to FAISS
    vector_store = FAISS.from_texts(texts, embedding_model, metadatas=metadatas or [])
    return vector_store

# Search in FAISS Vector Store
def search_vector_store(vector_store, query, embedding_model, top_k=5):
    results = vector_store.similarity_search(query, k=top_k)
    return results

# Formatting the input texts for embedding.
def format_data(data, type):
    texts, metadatas = [], []
    for item in data['results']['bindings']:
        label = item[f'{type}Label']['value']
        uri = item[f'{type}']['value']
        description = item[f'{type}Description']['value'] if f'{type}Description' in item else ""
        texts.append(label)
        metadatas.append({"label": label, "id": uri.split('/')[-1], "description": description})
    return texts, metadatas

def linkClasses(URI):
    for s, p, o in g.triples((None, RDF.type, URI)):
        style_uri = s
        style_name = g.value(s, SCHEMA.name)
        if style_name is None:
            continue
        results = vector_store.similarity_search(str(style_name), k=1)

        if results:
            match = results[0]
            wikidata_uri = URIRef(wd[match.metadata['id']])

            # Add the owl:sameAs link
            g.add((style_uri, OWL.sameAs, wikidata_uri))
            print(f"Linked: {style_name} -> {match.page_content} ({match.metadata['id']})")
        else:
            print(f"No match found for: {style_name}")



In [6]:
wikidata_data = query_wikidata(query_danceStyles)
print(wikidata_data)

{'head': {'vars': ['danceStyle', 'danceStyleLabel', 'danceStyleDescription']}, 'results': {'bindings': [{'danceStyle': {'type': 'uri', 'value': 'http://www.wikidata.org/entity/Q6658'}, 'danceStyleLabel': {'xml:lang': 'en', 'type': 'literal', 'value': 'Bamboula'}, 'danceStyleDescription': {'xml:lang': 'en', 'type': 'literal', 'value': 'kind of drum'}}, {'danceStyle': {'type': 'uri', 'value': 'http://www.wikidata.org/entity/Q9764'}, 'danceStyleLabel': {'xml:lang': 'en', 'type': 'literal', 'value': 'flamenco'}, 'danceStyleDescription': {'xml:lang': 'en', 'type': 'literal', 'value': 'genre of Spanish music and dance'}}, {'danceStyle': {'type': 'uri', 'value': 'http://www.wikidata.org/entity/Q14919'}, 'danceStyleLabel': {'xml:lang': 'en', 'type': 'literal', 'value': 'air guitar'}, 'danceStyleDescription': {'xml:lang': 'en', 'type': 'literal', 'value': 'form of dance and movement'}}, {'danceStyle': {'type': 'uri', 'value': 'http://www.wikidata.org/entity/Q17237'}, 'danceStyleLabel': {'xml:la

In [7]:
# Link Dance
wikidata_data_danceStyle = query_wikidata(query_danceStyles)
wikidata_data_instrument = query_wikidata(query_instrument)
text_danceStyle, metadata_danceStyle = format_data(wikidata_data_danceStyle, "danceStyle")
text_instrument, metadata_instrument = format_data(wikidata_data_danceStyle, "danceStyle")

In [8]:
model_name = "BAAI/bge-large-en-v1.5"
embedding_model = HuggingFaceEmbeddings(model_name=model_name, model_kwargs={'device': 'cpu'})
vector_store = FAISS.from_texts(text_danceStyle, embedding_model, metadatas=metadata_danceStyle)


In [9]:
wd = Namespace("http://www.wikidata.org/entity/")
DANCE = Namespace("http://example.org/dance/")
SCHEMA = Namespace("http://schema.org/")

linkClasses(DANCE.DanceStyle)



Linked: American Rhythm -> American Rhythm (Q42177945)
Linked: American Tribal Style -> American Tribal Style Belly Dance (Q448706)
Linked: Argentine tango -> Argentine tango (Q25116)
Linked: Baladi -> baladi (Q804596)
Linked: Balboa -> Balboa (Q804690)
Linked: Ballet -> ballet (Q41425)
Linked: Baroque dance -> Baroque dance (Q3015601)
Linked: Bhangra -> bhangra (Q13479601)
Linked: Bharatanatyam -> Bharatanatyam (Q334156)
Linked: Big Apple -> Big Apple (Q4904933)
Linked: Black Bottom -> Black Bottom (Q608451)
Linked: Blues dance -> blues dance (Q886025)
Linked: Bollywood dance -> Bollywood dance (Q891476)
Linked: Boogie-woogie -> boogie-woogie (Q892964)
Linked: Breaking -> popping (Q937440)
Linked: Bugg -> bugg (Q1002441)
Linked: Bunny hop -> bunny hop (Q4997884)
Linked: Carolina Shag -> Carolina shag (Q5044924)
Linked: Cham dance -> Cham dance (Q138567)
Linked: Charleston -> charleston (Q1066703)
Linked: Circle dance -> circle dance (Q17080806)
Linked: Collegiate Shag -> collegiate sh

In [10]:
# Link instruments

text_instrument, metadata_instrument = format_data(wikidata_data_instrument, "instrument")

model_name = "BAAI/bge-large-en-v1.5"
embedding_model = HuggingFaceEmbeddings(model_name=model_name, model_kwargs={'device': 'cpu'})
vector_store = FAISS.from_texts(text_instrument, embedding_model, metadatas=metadata_instrument)


In [11]:
wd = Namespace("http://www.wikidata.org/entity/")
DANCE = Namespace("http://example.org/dance/")
SCHEMA = Namespace("http://schema.org/")

linkClasses(DANCE.Instrument)



In [12]:
# Link dance practitioners


wikidata_data_dancers = query_wikidata(query_dance_practitioner)
text_dancers, metadata_dancers = format_data(wikidata_data_dancers, "musicalGroup")

model_name = "BAAI/bge-large-en-v1.5"
embedding_model = HuggingFaceEmbeddings(model_name=model_name, model_kwargs={'device': 'cpu'})
vector_store = FAISS.from_texts(text_dancers, embedding_model, metadatas=metadata_dancers)


In [13]:
wd = Namespace("http://www.wikidata.org/entity/")
DANCE = Namespace("http://example.org/dance/")
SCHEMA = Namespace("http://schema.org/")

linkClasses(DANCE.Practitioner)



Linked: Alina Kabaeva -> Alisa (Q2837199)
Linked: Anna Pavlova -> ANNA (Q1824889)
Linked: B-boy crews -> Hip Hop Boyz (Q1036597)
Linked: B-boys and B-girls -> Boys Like Girls (Q655648)
Linked: Ballroom professionals -> The Professionals (Q1515331)
Linked: Big Mijo -> Mr. Big (Q822802)
Linked: Bob Fosse -> Bob hund (Q291878)
Linked: Boogaloo Sam -> Sam and the Womp (Q2115414)
Linked: Carlos Gardel -> il Gardellino (Q3148454)
Linked: Carmen Amaya -> Carmen (Q1043600)
Linked: Carnival performers -> Carnival of Souls (Q1044088)
Linked: Charlie White -> Charlie (Q1066788)
Linked: Church dance ministries -> Dance (Q1507109)
Linked: Club dancers -> In The Club (Q3149663)
Linked: Contemporary bachata artists -> Pablo Bachata (Q3359958)
Linked: Court dancers -> Courteeners (Q1352997)
Linked: Cuban dance troupes -> Afro-Cuban All Stars (Q387977)
Linked: Dance sport champions -> Champion (Q1061186)
Linked: Don Campbell -> Don Huonot (Q627203)
Linked: Early music ensembles -> Ensemble Cinquecento 

In [14]:
# Link music genre
wikidata_data_music_genre = query_wikidata(query_dance_music_genre)
text_music_genre, metadata_music_genre = format_data(wikidata_data_music_genre, "music_genre")

model_name = "BAAI/bge-large-en-v1.5"
embedding_model = HuggingFaceEmbeddings(model_name=model_name, model_kwargs={'device': 'cpu'})
vector_store = FAISS.from_texts(text_music_genre, embedding_model, metadatas=metadata_music_genre)


In [15]:
wd = Namespace("http://www.wikidata.org/entity/")
DANCE = Namespace("http://example.org/dance/")
SCHEMA = Namespace("http://schema.org/")

linkClasses(DANCE.MusicGenre)



In [16]:
g.serialize(destination="../graphs/KG_base_dance_linked.ttl", format="turtle")
print("Graph saved to KG_base_dance_linked.ttl")


Graph saved to KG_base_dance_linked.ttl
